# `pt.layout_diagram` — see what plotlet did

`pt.layout_diagram(c)` returns a Chart leaf that composes via `|` /
`/` like any other chart. The diagram shows panel bounds, data
areas, gaps, and — overlaid as translucent rects — every chrome
region plotlet emits (panel, spines, title, axis labels, ticks,
legend). `pt.regions(c)` returns the same information as a list of
`{kind, bbox, name, meta}` dicts for programmatic checks.


In [1]:
import plotlet as pt

### 1. Single chart — scatter with continuous color + varying sizes

`scatter(color=<numeric col>, size=...)` triggers two legend aesthetics: a colorbar
for the continuous color mapping and a graded-dot size legend for `size`.
The diagram on the right shows the chrome regions for both, plus
the panel, spines, ticks, and axis labels.

In [2]:
import numpy as np
rng = np.random.default_rng(42)
n = 50
data = {
    'x': rng.standard_normal(n).tolist(),
    'y': (rng.standard_normal(n) * 0.5).tolist(),
    'mag':   [abs(z) * 80 + 10 for z in rng.standard_normal(n)],
}
data['y'] = [yy + 0.6 * xx for xx, yy in zip(data['x'], data['y'])]
data['val'] = data['y']

sc = pt.chart(title='scatter: color + size aesthetics',
              xlabel='x', ylabel='y', legend=True,
              data_width=380, data_height=240)
sc.scatter(data=data, x='x', y='y', color='val', size='mag', cmap='viridis')
sc.xticks(rotation=45, format="{:.7f}")
sc | pt.layout_diagram(sc)

### 2. Multi-chart layout with a shared legend

`pt.legend(a, b)` harvests entries from each source chart and
renders one legend leaf next to the panels. The diagram shows
every panel's chrome plus the legend leaf in its own frame — the
translate-stack carries per-leaf offsets so overlays land at the
right pixel everywhere.


In [3]:
data_a = {
    'x':      [1,2,3,4,5, 1,2,3,4,5],
    'y':      [1,4,2,5,3, 2,3,3,4,5],
    'series': ['trend']*5 + ['baseline']*5,
}
data_b = {
    'x':      [1,2,3,4,5, 1,2,3,4,5],
    'y':      [3,1,4,2,5, 4,2,3,3,4],
    'series': ['trend']*5 + ['baseline']*5,
}
a = pt.chart(title='Group A', xlabel='x', ylabel='y',
             data_width=220, data_height=160)
a.line(data=data_a, x='x', y='y', color='series')
b = pt.chart(title='Group B', xlabel='x', ylabel='y',
             data_width=220, data_height=160)
b.line(data=data_b, x='x', y='y', color='series')
layout = (a | b) | pt.legend(a, b)
layout / pt.layout_diagram(layout)


### 3. Attachments — annotation tracks alongside a main chart

`host.attach_above(track) / .attach_below() / .attach_left() /
.attach_right()` reserves margin space for an auxiliary chart that
shares the host's axis. The composite is one block to peer
composition: `attached` below is still a single `Chart` you can
pipe with `|` / `/`. The diagram shows the host's panel plus each
attached side as its own panel, with the inter-panel gap detected
and hatched.


In [4]:
data       = {'x': [1,2,3,4,5,6,7], 'y': [3,5,2,8,4,6,7]}
top_data   = {'x': [1,2,3,4,5,6,7], 'y': [1,2,1,2,1,2,1]}
right_data = {'x': [10,12,11,15,10,12,14], 'y': [1,2,3,4,5,6,7]}

main = pt.chart(title='daily count', xlabel='day', ylabel='count',
                data_width=300, data_height=180)
main.scatter(data=data, x='x', y='y')

top = pt.chart(ylabel='weekend', data_width=300, data_height=40)
top.line(data=top_data, x='x', y='y')

right = pt.chart(xlabel='rank', data_width=40, data_height=180)
right.line(data=right_data, x='x', y='y')

attached = main.attach_above(top).attach_right(right)
attached | pt.layout_diagram(attached)
